# 
The previous notebooks covered the "happy path" of  modeling data, replicating it, and storing it. But real-world systems run into edge cases:Cassandra 

- What if two clients try to register the same username at the same time?
- How do you automatically expire session tokens after 24 hours?
- What happens if you use Cassandra like a relational database (spoiler: bad things)?

This notebook covers the features and **anti-patterns** you need to know to run Cassandra in production without shooting yourself in the foot.

## Learning Objectives

By the end of this notebook, you'll understand:
- **Lightweight Transactions ( `IF NOT EXISTS` and compare-and-set (Paxos-based)LWT)** 
- **TTL (Time-To- auto-expiring dataLive)** 
- ** the one mutable data type in CassandraCounters** 
- ** when to use them, and why `BATCH` is NOT a performance trickBatches** 
- **Secondary  the feature you should almost never useindexes** 
- ** reading millions of rows without blowing up memoryPaging** 
- **Production tuning  settings that matter in real deploymentstips** 


## 
Make sure the cluster is running:

```bash
cd 03-technologies/databases/cassandra
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
 "Reload Window".


In [ ]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement, BatchStatement, BatchType
from cassandra import ConsistencyLevel
from tabulate import tabulate
import time

cluster = Cluster(["localhost"], port=9042)
session = cluster.connect()

session.execute("""
    CREATE KEYSPACE IF NOT EXISTS demo
    WITH REPLICATION = { 'class': 'SimpleStrategy', 'replication_factor': 3 }
""")
session.set_keyspace('demo')

print(f"Connected to: {cluster.metadata.cluster_name}")


## Step 1: Lightweight Transactions ( Compare-and-SetLWT) 

Cassandra writes are normally "last write  if two clients write the same key at the same time, Cassandra keeps whichever has the highest timestamp. That's fine for most data, but sometimes you need **uniqueness** or **conditional updates**.wins" 

### The Problem
Imagine a signup flow:
 No
 No
 Alice`
 Bob`

Now both Alice and Bob think they own ` a **race condition**.cooluser` 

### The Solution: LWT
Cassandra supports `IF NOT EXISTS` and `IF condition` clauses. Under the hood, these use the **Paxos consensus algorithm** across replicas to guarantee the check-and-set is atomic.

 **LWT is ~4>  slower** than a normal write because it requires 4 round-trips between replicas. Use it only when you truly need uniqueness or conditional writes.


In [ ]:
# Create a table for usernames: we want username to be globally unique
session.execute("""
    CREATE TABLE IF NOT EXISTS users_by_username (
        username text PRIMARY KEY,
        user_id bigint,
        email text
    )
""")

# First signup succeeds
result = session.execute("""
    INSERT INTO users_by_username (username, user_id, email)
    VALUES ('cooluser', 1, 'alice@example.com')
    IF NOT EXISTS
""").one()

print(f"Alice's signup: applied = {result.applied}")
# The driver returns [applied] and, if not applied, the current row values.

# Second signup with the SAME username fails (LWT detected the conflict)
result = session.execute("""
    INSERT INTO users_by_username (username, user_id, email)
    VALUES ('cooluser', 2, 'bob@example.com')
    IF NOT EXISTS
""").one()

print(f"Bob's signup:   applied = {result.applied}")
print(f"                existing user_id = {result.user_id}, email = {result.email}")
print()
print("LWT guaranteed only ONE signup succeeded, even under concurrency.")


In [ ]:
# Conditional UPDATE: only change email if the current one matches what we expect
# (Useful for optimistic  like a CAS compare-and-swap.)locking 
result = session.execute("""
    UPDATE users_by_username
    SET email = 'alice+new@example.com'
    WHERE username = 'cooluser'
    IF email = 'alice@example.com'
""").one()

print(f"Update applied: {result.applied}")

# Try the same update  it will fail because the email already changedagain 
result = session.execute("""
    UPDATE users_by_username
    SET email = 'alice+newer@example.com'
    WHERE username = 'cooluser'
    IF email = 'alice@example.com'
""").one()

print(f"Second update applied: {result.applied}")
print(f"Current email in DB:   {result.email}")


### LWT Consistency  `SERIAL`Level 

Normal writes use `LOCAL_QUORUM` or similar. LWT uses a special consistency level called `SERIAL` (or `LOCAL_SERIAL` within one datacenter). This is what makes the Paxos rounds work.

**Rule of thumb:**
- Use LWT for truly unique things: usernames, emails, reservation tokens, inventory decrements
- Do NOT use LWT for everyday  it's expensiveupdates 

### When NOT to use LWT
- As a general "update only if" for every write (kills performance)
- For counters (use native `counter` columns instead)
- For high-throughput writes (LWT doesn't scale like normal writes)


## Step 2:  Auto-Expiring DataTTL 

Cassandra can automatically delete data after a set number of seconds. This is called **TTL (Time-To-Live)**. TTL is perfect for:

- **Session  expire after 24 hourstokens** 
- **Email verification  expire after 15 minutescodes** 
- **Time-series  drop metrics older than 30 daysdata** 
- ** discard stale dataCaches** 

When a row's TTL expires, it becomes a tombstone (remember Notebook 4!) and compaction cleans it up later. With `TimeWindowCompactionStrategy`, whole SSTables can drop at  very efficient.once 


In [ ]:
# Session tokens that auto-expire after 5 seconds (for demo purposes)
session.execute("""
    CREATE TABLE IF NOT EXISTS session_tokens (
        token text PRIMARY KEY,
        user_id bigint,
        created_at timestamp
    )
""")

from datetime import datetime

# Insert with a 5-second TTL
session.execute("""
    INSERT INTO session_tokens (token, user_id, created_at)
    VALUES ('abc123', 42, %s)
    USING TTL 5
""", (datetime.now(),))

print("Inserted token 'abc123' with TTL=5s\n")

# Read it right after insert: still there
row = session.execute(
    "SELECT token, user_id, TTL(user_id) AS remaining FROM session_tokens WHERE token = 'abc123'"
).one()
print(f"Right after insert: token={row.token}, user_id={row.user_id}, remaining TTL={row.remaining}s")

print("\nWaiting 6 seconds for TTL to expire...")
time.sleep(6)

# Read again: gone
row = session.execute("SELECT * FROM session_tokens WHERE token = 'abc123'").one()
print(f"\nAfter 6 seconds: {row}")
print("\nTTL expired, the row is gone. No DELETE statement needed.")


### Setting a Default TTL at the Table Level

Instead of repeating `USING TTL N` on every insert, you can set a default on the table:

```sql
CREATE TABLE audit_log (...)
WITH default_time_to_live = 2592000;  -- 30 days
```

### TTL Gotchas

1. **TTLs create tombstones.** Deleting millions of rows via TTL still leaves millions of tombstones until compaction clears them.
2. **Pair high-TTL churn with TWCS.** TimeWindowCompactionStrategy drops entire expired SSTables without needing to merge them. Much cheaper than STCS or LCS.
3. **You cannot extend an existing row's  a new write with a longer TTL replaces the whole row's TTL only for the columns written.TTL** 


## Step 3:  The One Mutable TypeCounters 

Regular Cassandra columns are immutable in  an UPDATE is really an INSERT of a new version. That makes `col = col + 1` unsafe across concurrent writers.spirit 

For that, Cassandra has **counter columns**:

```sql
UPDATE page_views SET views = views + 1 WHERE page_id = 'home';
```

### Counter Rules
- A counter table can contain **only** counter columns and the primary key.
- You cannot INSERT into a counter  only UPDATE.table 
- You cannot set a counter to an absolute  only increment or decrement.value 
- Counters use a special replication protocol under the hood (slower than normal writes).


In [ ]:
# Counter table, e.g. page view counts
session.execute("""
    CREATE TABLE IF NOT EXISTS page_views (
        page_id text PRIMARY KEY,
        views counter
    )
""")

# Increment a counter (you cannot INSERT into a counter table, only UPDATE)
for _ in range(5):
    session.execute("UPDATE page_views SET views = views + 1 WHERE page_id = 'home'")

session.execute("UPDATE page_views SET views = views + 10 WHERE page_id = 'home'")
session.execute("UPDATE page_views SET views = views - 2 WHERE page_id = 'home'")

row = session.execute("SELECT page_id, views FROM page_views WHERE page_id = 'home'").one()
print(f"page_id={row.page_id}, views={row.views}")
print("\nCounter safely accumulates concurrent increments.")


### When NOT to Use Counters
- **You need exact correctness under failure.** Counters are NOT  if a client retries a failed increment, the counter may be incremented twice. For billing/money use a real ledger (append-only transactions).idempotent 
- **You need to read the current value and write a derived one.** That's a CAS, not a  use LWT.counter 

In practice, counters are best for analytics-style metrics where "approximately right" is good enough (view counts, like counts, etc.).

## Step 4:  When They Help (and When They Hurt)Batches 

CQL has a `BATCH` statement:

```sql
BEGIN BATCH
    INSERT INTO users (...) VALUES (...);
    INSERT INTO users_by_email (...) VALUES (...);
APPLY BATCH;
```

 The #1 Cassandra Anti-Pattern### 
Developers coming from SQL assume `BATCH = faster` (like a transaction). **In Cassandra, that's exactly backwards.**

- A batch is sent to a **single coordinator node**.
- The coordinator must forward each statement to the right partition owner.
- Large batches overload the coordinator and slow down writes for EVERYONE.

**The golden rule:** Only batch writes that go to the **same partition**. These are called *logged batches* but for same-partition work, the underlying driver actually turns them into atomic single-partition batches, which ARE efficient.


In [ ]:
# GOOD: batch writes that touch ONE partition (atomic, efficient)
session.execute("""
    CREATE TABLE IF NOT EXISTS order_items (
        order_id bigint,
        item_id int,
        name text,
        qty int,
        PRIMARY KEY (order_id, item_id)
    )
""")

batch = BatchStatement(batch_type=BatchType.LOGGED)
for item_id, name, qty in [(1, 'Laptop', 1), (2, 'Mouse', 2), (3, 'Keyboard', 1)]:
    batch.add(SimpleStatement(
        "INSERT INTO order_items (order_id, item_id, name, qty) VALUES (%s, %s, %s, %s)"
    ), (5001, item_id, name, qty))

session.execute(batch)
print("Same-partition batch: all 3 items inserted atomically for order 5001")

rows = session.execute("SELECT * FROM order_items WHERE order_id = 5001")
for r in rows:
    print(f"   item {r.item_id}: {r.name} x {r.qty}")


### BAD Batch Example (don't do this)

```python
 DON'T: batching unrelated partitions to "save round-trips"# 
batch = BatchStatement()
for user_id in range(10000):
    batch.add(SimpleStatement(
        "INSERT INTO users (user_id, name) VALUES (%s, %s)"
    ), (user_id, f'user-{user_id}'))
```session.execute(batch)  # 

Instead, use **async parallel inserts**:

```python
#  DO: fire-and-collect async inserts
futures = [session.execute_async(stmt, params) for params in many_rows]
for f in futures:
    f.result()
```

### Summary
| Batch Type | When to use |
|-----------|-------------|
| Same-partition LOGGED batch | Atomic multi-row update to one  safe and fast |partition 
| UNLOGGED batch (same partition) | When atomicity isn't required, marginally less overhead |
| Multi-partition batch | Only for rare cross-table atomicity needs; otherwise avoid |


## Step 5: Secondary  Usually a TrapIndexes 

Cassandra supports `CREATE INDEX`. It looks like a PostgreSQL index, but it is fundamentally different:

- A secondary index is **distributed across ALL nodes**.
- A query by indexed column must ask **every  a full cluster fan-out.node** 
- Cardinality matters: low-cardinality columns (e.g., a `status` with 3 values) create huge index partitions (hotspots); high-cardinality columns (e.g., `email`) don't benefit much either.

### The Right Alternative
**Build a second table** with the desired partition key (query-driven modeling from Notebook 1). Yes, duplicate the  storage is cheap, slow queries are expensive.data 


In [ ]:
# Demonstration (don't do this in production for real lookups)
session.execute("""
    CREATE TABLE IF NOT EXISTS users_full (
        user_id bigint PRIMARY KEY,
        username text,
        email text,
        country text
    )
""")

# Cassandra lets you create a secondary  but bewareindex 
session.execute("CREATE INDEX IF NOT EXISTS ON users_full (email)")
print("Created secondary index on users_full(email)")
print("\nQuerying by email will now work, but every such query fans out to ALL nodes.")
print("For anything high-traffic, build a users_by_email table with email as partition key instead.")


### A Note on SASI and SAI
Cassandra has newer index implementations (SASI, SAI in 5.0) that fix some issues. They're still not a substitute for good data modeling in high-throughput systems, but they're worth knowing about. The "build a second table" approach is the safest default and what Discord/Netflix/Apple actually do at scale.


## Step 6:  Reading Large Result SetsPaging 

When a query matches millions of rows, you don't want to load them all into memory. The Python driver automatically **pages** results:

- `fetch_ number of rows per page (default: 5000)size` 
- The driver fetches the next page only when you iterate past the current one.

```python
statement = SimpleStatement("SELECT * FROM big_table", fetch_size=1000)
for row in session.execute(statement):
    process(row)  # driver fetches new pages transparently
```

You can also **manually control paging** (useful for web APIs that return cursors):

```python
result = session.execute(statement)
paging_state = result.paging_state   # opaque  send to client as cursorbytes 

# Next call (maybe from a new HTTP request)
result = session.execute(statement, paging_state=paging_state)
```


In [ ]:
# Quick paging demo
session.execute("""
    CREATE TABLE IF NOT EXISTS paging_demo (
        bucket int,
        id int,
        data text,
        PRIMARY KEY (bucket, id)
    )
""")

insert = session.prepare("INSERT INTO paging_demo (bucket, id, data) VALUES (?, ?, ?)")
for i in range(50):
    session.execute(insert, (1, i, f'row-{i}'))

stmt = SimpleStatement("SELECT * FROM paging_demo WHERE bucket = 1", fetch_size=10)
result = session.execute(stmt)

print(f"First page fetched with fetch_size=10")
print(f"Has more pages? {result.has_more_pages}")

count = 0
for _ in result:
    count += 1
print(f"\nIterated {count} rows (driver fetched subsequent pages automatically)")


## Step 7: Production Tuning Tips

A condensed list of things people wish they'd known before rolling out Cassandra:

### Hardware & OS
- **SSDs are mandatory.** LSM compaction does heavy sequential + random I/O. Spinning disks will melt.
- **Separate commit log and data disks** if possible.
- **Disable swap.** Cassandra misbehaves badly when paged out.
- **Heap size**: 16 GB. Bigger heaps = longer GC pauses. Use G1GC in Cassandra 4+.8

### Replication & Topology
- Use **`NetworkTopologyStrategy`** for every keyspace (not just  even dev).production 
- **RF=3** is the standard. RF=1 is a "data-loss-is-fine" setup only.
- Spread nodes across racks/AZs so a rack failure only loses one replica.

### Data Modeling
- **Target partition size under 100 MB.** Over ~1 GB and things get painful.
- **Max rows per partition around 100,000** is a useful rule of thumb.
- **Don't use `ALLOW FILTERING`** in production code  it's an "I know this is slow" flag.paths 
- **Denormalize freely.** Disks are cheap; fan-out queries are not.

### Operations
- Run **`nodetool repair`** regularly (at least within `gc_grace_seconds`) to prevent tombstone resurrection. Use **`reaper`** or similar tooling to automate.
- Monitor **pending compactions, dropped mutations, read/write latency p99, and tombstone warnings** in logs.
- Add nodes **one at a  each addition triggers streaming and gossip churn.time** 
- When decommissioning, use `nodetool  don't just turn the box off.decommission` 

### Consistency
- Default to `LOCAL_QUORUM` read + `LOCAL_QUORUM` write for strong consistency within a DC.
- Use `LOCAL_ONE` for analytics/background jobs where staleness is fine.
- Avoid `CL=ALL` in  one slow node stalls every query.production 

### Driver
- **Prepare statements once, reuse  `session.prepare(...)`. Prepared statements are cached server-side and avoid re-parsing.forever** 
- Set **token-aware + DC-aware load balancing** (default in modern drivers).
- Use **async execution** for parallel  don't loop sync `execute()` calls.writes 


## Step 8: Quick  Anti-Pattern Cheat SheetReference 

 Anti-Pattern What to do instead | | | 
|-----------------|----------------------|
| `SELECT * FROM t WHERE non_pk = x ALLOW FILTERING` | Create a second table keyed by that column |
| Large multi-partition `BATCH` | Fire async parallel inserts |
| Secondary index on high-traffic column | Second table with that column as partition key |
| Unbounded-growth partition (single channel_id) | Add a time bucket to the partition key |
| Using `CL=ALL` everywhere | Use `LOCAL_QUORUM`, reserve ALL for rare admin tasks |
| Running without `nodetool repair` | Schedule repairs inside `gc_grace_seconds` |
| Treating counters as exact ledgers | Use an append-only event table + materialize totals |
| `DELETE` entire partitions every minute | Use TTL +  whole SSTables drop for free |TWCS 
| Reads that return millions of rows at once | Use `fetch_size` paging or cursor (`paging_state`) |
| Using LWT for every update | Use LWT only for true uniqueness/ it's 4Cas slower | 


## 
1. **LWT** (`IF NOT EXISTS`, `IF condition`) gives you linearizable compare-and- use sparingly, it's expensive.set 

2. **TTL** auto-expires rows. Pair with TWCS for cheap, bulk data expiry.

3. **Counters** are the only mutable column type. They're not exact under  good for view counts, bad for money.retries 

4. ** transactions.** Use them only for same-partition atomicity, never for "multi-insert performance."Batches 

5. **Secondary indexes are a trap** for high-traffic queries. Build a second table keyed by the column instead.

6. **Paging** lets you stream large result sets without blowing up memory.

7. **Production Cassandra** needs: SSDs, `NetworkTopologyStrategy`, `nodetool repair`, monitoring, `LOCAL_QUORUM`, async driver calls, and disciplined data modeling.

## 
You've now covered Cassandra end-to-end:
1. Partition keys & query-driven data modeling
2. Clustering columns & wide rows
3. Replication & tunable consistency
4. LSM storage & compaction strategies
5. **LWT, TTL, counters, batches, anti-patterns, and production tuning** (this notebook)

You're ready to reason about Cassandra in system design interviews and to avoid the most common production pitfalls.


In [ ]:
cluster.shutdown()
print("Connection closed.")
